In [4]:
%load_ext autoreload
%autoreload 2


from libthesis import pdf_writer, update_layout

In [10]:
import pandas
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from libthesis import update_layout, pdf_writer


def generate_chart(kmer : int = 10):
    # Load data
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
    df = df[df["kmer_size"] == kmer].copy()
    df = df[
        df["madb_distance"].notnull() &
        df["cogent3_pd"].notnull()
    ].copy()

    # Regression (for R² and RSS only)
    X = df["cogent3_pd"].values.reshape(-1, 1)
    y = df["madb_distance"].values
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    rss = ((y - y_pred) ** 2).sum()

    # Base scatter plot
    fig = px.scatter(
        df,
        x="cogent3_pd",
        y="madb_distance",
        color="species",
        symbol="madb_cycles",
        symbol_map={True: "x", False: "circle"},
        hover_data=["unique_id"],
        opacity=0.6
    )

    # Parity line
    fig.add_trace(go.Scatter(
        x=[0, 0.1],
        y=[0, 0.1],
        mode="lines",
        line=dict(dash="dot", color="gray"),
        showlegend=False
    ))

    # R² + RSS annotation
    fig.add_annotation(
        text=f"R² = {r2:.3f}<br>RSS = {rss:.2e}",
        xref="paper", yref="paper",
        x=0.98, y=0.02,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=14)
    )

    fig.update_xaxes(range=[0, 0.1])
    fig.update_yaxes(range=[0, 0.1], scaleanchor="x", scaleratio=1)
    
    # Layout
    update_layout(fig, x_title="$PD_{aligned}$", y_title="$PD_{braids}$", in_panel=True)

    # Save
    writer = pdf_writer()
    writer(fig, "madb_distance_vs_cogent3_distance")
    fig.show()

generate_chart(10)

In [8]:
import pandas
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.metrics import mean_squared_error

def generate_jaccard_chart(kmer : int = 10):
    # Load data
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
    
    df = df[df["kmer_size"] == kmer].copy()
    df = df[df["jaccard_distance"].notnull() & df["cogent3_pd"].notnull()].copy()
    df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})

    # Prepare data
    X = df["cogent3_pd"].values
    Y = df["jaccard_distance"].values

    # LOWESS smoothing
    lowess_result = lowess(Y, X, frac=0.3, return_sorted=True)  # adjust `frac` for smoothing level
    x_fit, y_fit = lowess_result[:, 0], lowess_result[:, 1]

    # Interpolate to original X for RSS
    y_interp = np.interp(X, x_fit, y_fit)
    rss = np.sum((Y - y_interp) ** 2)
    r2 = 1 - rss / np.sum((Y - Y.mean()) ** 2)

    # Scatter plot
    fig = px.scatter(
        df,
        x="cogent3_pd",
        y="jaccard_distance",
        color="species",
        symbol="cycle_status",
        hover_data=["unique_id"],
        opacity=0.6
    )

    # Add LOWESS curve
    fig.add_trace(
        go.Scatter(
            x=x_fit,
            y=y_fit,
            mode="lines",
            line=dict(color="black", dash="dash"),
            name="LOWESS fit",
            showlegend=False
        )
    )

    # Add annotation
    fig.add_annotation(
        text=f"R² = {r2:.3f}<br>RSS = {rss:.3e}",
        xref="paper", yref="paper",
        x=0.98, y=0.02,
        showarrow=False,
        xanchor="right", yanchor="bottom",
        font=dict(size=14)
    )

    fig.update_xaxes(range=[0, 0.1])

    update_layout(fig, in_panel=True, x_title="$PD_{aligned}$", y_title="$Jaccard \\text{ distance}$")
    fig.show()
    write_pdf = pdf_writer()
    write_pdf(fig, "jaccard_distance_vs_cogent3_distance_lowess")
generate_jaccard_chart(15)